# Dataset Reconnaissance
Inspect the structure of each mounted dataset **without extracting**.
Run this once, push results to HuggingFace, then never re-run.

In [ ]:
# Cell 1: Imports (no pip install needed — uses only stdlib + zipfile)
import os, json, zipfile
from pathlib import Path
from collections import Counter

# Try rarfile, but don't fail if missing
try:
    import rarfile
    HAS_RAR = True
except ImportError:
    HAS_RAR = False
    print("NOTE: rarfile not available — RAR archives will be skipped")

REPO_ID = "david-net-av/backup"
KAGGLE_INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/dataset_recon")
OUT_DIR.mkdir(exist_ok=True)

print(f"Setup done. RAR support: {HAS_RAR}")

In [ ]:
# Cell 2: Discover mounted datasets
mounted = []
for d in sorted(KAGGLE_INPUT.iterdir()):
    if not d.is_dir():
        continue
    subdirs = [s for s in d.iterdir() if s.is_dir()]
    if d.name == "datasets" and len(subdirs) > 1:
        print(f"{d.name}/ is a wrapper, checking inside...")
        for s in sorted(subdirs):
            mounted.append(s)
            print(f"  {s.name}/")
    else:
        mounted.append(d)
        print(f"  {d.name}/")

print(f"\nFound {len(mounted)} datasets.")

In [ ]:
# Cell 3: Recon functions
def recon_zip(path: Path) -> dict:
    try:
        with zipfile.ZipFile(path) as zf:
            entries = [{"name": i.filename, "size": i.file_size, "compressed": i.compress_size}
                       for i in zf.infolist()]
            return {"type": "zip", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "zip", "error": str(e)}


def recon_rar(path: Path) -> dict:
    if not HAS_RAR:
        return {"type": "rar", "error": "rarfile module not available"}
    try:
        rf = rarfile.RarFile(path)
        entries = [{"name": i.filename, "size": i.file_size,
                    "compressed": getattr(i, 'compress_size', 0)} for i in rf.infolist()]
        return {"type": "rar", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "rar", "error": str(e)}


def build_tree(entries):
    tree, exts, total_size = {}, Counter(), 0
    for e in entries:
        total_size += e.get("size", 0)
        exts[Path(e["name"]).suffix.lower()] += 1
        node = tree
        for p in e["name"].split("/")[:-1]:
            node = node.setdefault(p, {})
    return {"tree": tree, "extensions": dict(exts), "total_size": total_size, "file_count": len(entries)}


def recon_archive(path):
    ext = path.suffix.lower()
    if ext == ".zip":
        return recon_zip(path)
    elif ext in (".rar", ".001"):
        return recon_rar(path)
    return {"type": "unknown", "error": f"unsupported: {ext}"}

print("Recon functions ready.")

In [ ]:
# Cell 4: Run recon on all mounted datasets
results = {}

for ds_dir in mounted:
    ds_name = ds_dir.name
    print(f"\n=== {ds_name} ===")
    
    archives, loose_files = [], []
    try:
        for f in ds_dir.rglob("*"):
            if not f.is_file():
                continue
            is_archive = (
                f.suffix.lower() in (".zip", ".rar") or
                (f.suffix.isdigit() and len(f.suffix) == 3 and f.suffix != "000")
            )
            if is_archive:
                archives.append(f)
            else:
                loose_files.append({"name": str(f.relative_to(ds_dir)), "size": f.stat().st_size})
    except Exception as e:
        print(f"  Error scanning: {e}")
        continue
    
    report = {"name": ds_name, "path": str(ds_dir), "archives": [],
              "loose_files_count": len(loose_files), "loose_files_sample": loose_files[:20]}
    
    for arch in sorted(archives, key=lambda p: p.name)[:5]:
        print(f"  Scanning: {arch.name} ({arch.stat().st_size / 1e6:.1f} MB)")
        recon = recon_archive(arch)
        if "entries" in recon:
            info = build_tree(recon["entries"])
            recon.update({"tree": info["tree"], "extensions": info["extensions"],
                         "total_size": info["total_size"], "file_count": info["file_count"]})
            print(f"    -> {info['file_count']} files, {info['total_size'] / 1e9:.2f} GB")
            print(f"    Extensions: {info['extensions']}")
        else:
            print(f"    -> {recon.get('error', 'unknown')}")
        report["archives"].append({"name": arch.name, "size": arch.stat().st_size, "recon": recon})
    
    if not archives and loose_files:
        print(f"  No archives. Analyzing {len(loose_files)} loose files...")
        top_dirs, exts, total_size = Counter(), Counter(), 0
        for lf in loose_files:
            parts = lf["name"].split("/")
            if len(parts) >= 2:
                top_dirs[parts[0]] += 1
            exts[Path(lf["name"]).suffix.lower()] += 1
            total_size += lf["size"]
        print(f"  Top-level: {dict(top_dirs)}")
        print(f"  Extensions: {dict(exts)}")
        print(f"  Total: {total_size / 1e9:.2f} GB")
        report["top_dirs"] = dict(top_dirs)
        report["extensions"] = dict(exts)
        report["total_size"] = total_size
    
    results[ds_name] = report
    print(f"  Archives: {len(archives)}, Loose: {len(loose_files)}")

print(f"\nRecon complete for {len(results)} datasets.")

In [ ]:
# Cell 5: Save and summarize
for name, report in results.items():
    out_path = OUT_DIR / f"{name}.json"
    with open(out_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"Saved: {out_path}")

print("\n=== SUMMARY ===")
for name, report in results.items():
    total = sum(a["recon"].get("total_size", 0) for a in report["archives"] if "recon" in a)
    print(f"{name}: {len(report['archives'])} archives, {total/1e9:.2f} GB, {report['loose_files_count']} loose files")

In [ ]:
# Cell 6: Push to HuggingFace
from huggingface_hub import HfApi

# Read HF_TOKEN from Kaggle Secrets (not env var)
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("hf")

if not hf_token:
    print("ERROR: HF_TOKEN not found. Add it: Settings -> Secrets -> Add -> Name=HF_TOKEN")
else:
    api = HfApi(token=hf_token)
    try:
        api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
    except Exception as e:
        print(f"Repo note: {e}")
    for f in OUT_DIR.glob("*.json"):
        api.upload_file(path_or_fileobj=str(f), path_in_repo=f"dataset_recon/{f.name}",
                        repo_id=REPO_ID, repo_type="model")
        print(f"Pushed: {f.name}")
    print(f"Done — reports at {REPO_ID}/dataset_recon/")